# 11 CV_correction_per_fold_standardisation

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
frozen = pd.read_csv("FROZEN_candidate_pool.csv")
full_coverage = pd.read_csv("full_candidate_pool_final_coverage.csv")
frozen_genes = frozen["Gene"].tolist()

sub = full_coverage[full_coverage["Gene"].isin(frozen_genes)]
gene_to_col = {}
for _, r in sub.iterrows():
    gene_to_col[r["Gene"]] = r["Gene"] if r["TCGA_HNSC_Match"] == "Direct" else r["TCGA_HNSC_Alias_Used"]

genes = list(gene_to_col.keys())
tcga_cols = list(gene_to_col.values())
print("Candidate pool genes:", len(gene_to_col))

In [ ]:
usecols = ["Patient ID", "Sample ID", "Overall Survival (Months)", "Overall Survival Status"] + tcga_cols
df = pd.read_csv("clinical_rna_fire_combine.csv", usecols=usecols, engine="python")
df = df[df["Sample ID"].str.endswith("-01")].copy()

for c in tcga_cols:
    df[c] = np.log2(df[c].astype(float) + 1)

df["OS_months"] = df["Overall Survival (Months)"]
df["Event"] = df["Overall Survival Status"].astype(str).str.startswith("1").astype(int)
df = df.dropna(subset=["OS_months", "Event"])
df = df[df["OS_months"] > 0].reset_index(drop=True)

print("Derivation cohort: N =", df.shape[0], "| Events =", int(df["Event"].sum()))

X_raw = df[tcga_cols].copy()
X_raw.columns = genes
y = Surv.from_dataframe("Event", "OS_months", df)
print("Raw (unstandardised) design matrix:", X_raw.shape)

In [ ]:
X_allz = (X_raw - X_raw.mean()) / X_raw.std()
full_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01, n_alphas=100)
full_model.fit(X_allz.values, y)
alphas = full_model.alphas_
print(f"Alpha path: {len(alphas)} values, from {alphas.max():.5f} down to {alphas.min():.5f}")

In [ ]:
N_FOLDS = 10
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_scores = np.full((N_FOLDS, len(alphas)), np.nan)
fold_assignment = pd.Series(index=df.index, dtype=int)

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X_raw)):
    fold_assignment.iloc[test_idx] = fold_idx
    X_train_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    train_mean, train_std = X_train_raw.mean(), X_train_raw.std()
    X_train = (X_train_raw - train_mean) / train_std
    X_test = (X_test_raw - train_mean) / train_std

    try:
        fold_model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=alphas, alpha_min_ratio=0.01)
        fold_model.fit(X_train.values, y_train)
        for a_idx, alpha in enumerate(alphas):
            try:
                risk_scores = fold_model.predict(X_test.values, alpha=alpha)
                cv_scores[fold_idx, a_idx] = concordance_index_censored(y_test["Event"], y_test["OS_months"], risk_scores)[0]
            except Exception:
                continue
    except Exception as e:
        print(f"Fold {fold_idx} failed to fit: {e}")

    print(f"Fold {fold_idx+1}/{N_FOLDS} done -- train N={len(train_idx)}, test N={len(test_idx)} (standardised within fold only)")

In [ ]:
fold_level_df = pd.DataFrame(cv_scores, columns=[f"alpha_{i}_{a:.5f}" for i, a in enumerate(alphas)])
fold_level_df.index.name = "fold"
fold_level_df.to_csv("cv_fold_level_c_index_CORRECTED.csv")
print("Saved cv_fold_level_c_index_CORRECTED.csv:", fold_level_df.shape, "(10 folds x 100 alphas)")

fold_assignment.to_frame("cv_fold").to_csv("cv_patient_fold_assignment.csv")
print("Saved cv_patient_fold_assignment.csv -- which fold each of the 514 patients fell into")

fold_level_df.head()

In [ ]:
mean_cv = np.nanmean(cv_scores, axis=0)
se_cv = np.nanstd(cv_scores, axis=0) / np.sqrt(np.sum(~np.isnan(cv_scores), axis=0))

results_df = pd.DataFrame({"alpha": alphas, "mean_c_index": mean_cv, "se_c_index": se_cv})
results_df.to_csv("cv_results_CORRECTED.csv", index=False)

best_idx = np.nanargmax(mean_cv)
alpha_min = alphas[best_idx]
best_score = mean_cv[best_idx]
best_se = se_cv[best_idx]
print(f"Best CV C-index: {best_score:.4f} +/- {best_se:.4f} at alpha_min = {alpha_min:.5f}")

threshold = best_score - best_se
candidates_1se = [(a, m) for a, m in zip(alphas, mean_cv) if not np.isnan(m) and m >= threshold]
alpha_1se = max(candidates_1se, key=lambda x: x[0])[0]
print(f"alpha_1se (sparsest within 1 SE) = {alpha_1se:.5f}")

In [ ]:
X_full = (X_raw - X_raw.mean()) / X_raw.std()

def fit_and_report(alpha, label):
    model = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[alpha])
    model.fit(X_full.values, y)
    coefs = model.coef_.flatten()
    selected = [(g, c) for g, c in zip(genes, coefs) if c != 0]
    selected.sort(key=lambda x: -abs(x[1]))
    risk_scores = model.predict(X_full.values, alpha=alpha)
    cidx = concordance_index_censored(y["Event"], y["OS_months"], risk_scores)[0]
    print(f"{label}: {len(selected)} genes | in-sample C-index = {cidx:.3f}")
    result_df = pd.DataFrame(selected, columns=["Gene", "Coefficient"])
    result_df["HR_per_unit"] = np.exp(result_df["Coefficient"])
    result_df.to_csv(f"HNSC_hypoxia_signature_{label}_REVISED.csv", index=False)
    return result_df

df_min = fit_and_report(alpha_min, "lambda_min")
df_1se = fit_and_report(alpha_1se, "lambda_1se")

In [ ]:
locked_min = set(pd.read_csv("HNSC_hypoxia_signature_lambda_min.csv")["Gene"])
locked_1se = set(pd.read_csv("HNSC_hypoxia_signature_lambda_1se.csv")["Gene"])
revised_min = set(df_min["Gene"])
revised_1se = set(df_1se["Gene"])

print("=== COMPARISON: LOCKED vs REVISED (corrected per-fold CV) ===")
print(f"lambda.min -- locked: {len(locked_min)} genes, revised: {len(revised_min)} genes, identical: {locked_min == revised_min}")
print(f"lambda.1se -- locked: {len(locked_1se)} genes, revised: {len(revised_1se)} genes, identical: {locked_1se == revised_1se}")